<h1 style="text-align: center;">Language Models - Tutorial</h1> 

## Install dependencies (Optional)

In [1]:
!pip3 install transformers nltk

In [2]:
!pip3 install "numpy<2.0.0" --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 34.1 MB/s  0:00:00m0:00:010:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.


## N-Grams Using Brown Corpus

In [3]:
import nltk
from nltk.corpus import brown
from nltk.util import ngrams
from collections import Counter # used for frequency counting

# Download Brown corpus if not already downloaded
nltk.download('brown')

[nltk_data] Downloading package brown to /Users/fabienyah/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

### Basic stats

In [4]:
words = brown.words() # list of all tokens including punctuations
sents = brown.sents()
paras = brown.paras()
categories = brown.categories()

print(f"Number of words: {len(words)}")
print(f"Number of sentences: {len(sents)}")
print(f"Number of paragraphs: {len(paras)}")
print(f"Number of categories: {len(categories)}")
print(f"Categories: {categories}")
print(f"Vocabulary size: {len(set([w.lower() for w in words]))}")

Number of words: 1161192
Number of sentences: 57340
Number of paragraphs: 15667
Number of categories: 15
Categories: ['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']
Vocabulary size: 49815


In [5]:
import string

# Load and preprocess Brown corpus (convert to lowercase)
tokens = [word.lower() for word in words] # include punctuations  
clean_tokens = [w for w in tokens if w not in string.punctuation]

print("Done")

Done


### Unigrams

In [6]:
unigrams = list(ngrams(tokens, 1))
unigram_freq = Counter(unigrams)

print("Top 10 Unigrams:")
for word, freq in unigram_freq.most_common(10):
    print(f"{word}: {freq}")

Top 10 Unigrams:
('the',): 69971
(',',): 58334
('.',): 49346
('of',): 36412
('and',): 28853
('to',): 26158
('a',): 23195
('in',): 21337
('that',): 10594
('is',): 10109


In [7]:
unigram_freq = Counter(clean_tokens)

# Show top 10 most common unigrams (no punctuation)
print("Top 10 Unigrams (Punctuation Removed):")
for word, freq in unigram_freq.most_common(10):
    print(f"{word}: {freq}")

Top 10 Unigrams (Punctuation Removed):
the: 69971
of: 36412
and: 28853
to: 26158
a: 23195
in: 21337
that: 10594
is: 10109
was: 9815
he: 9548


### Bigrams

In [8]:
bigrams = list(ngrams(clean_tokens, 2))
bigram_freq = Counter(bigrams)

print("Top 10 Bigrams:")
for pair, freq in bigram_freq.most_common(10):
    print(f"{pair}: {freq}")

Top 10 Bigrams:
('of', 'the'): 9721
('in', 'the'): 6041
('to', 'the'): 3492
('on', 'the'): 2477
('and', 'the'): 2247
('for', 'the'): 1857
('to', 'be'): 1718
('at', 'the'): 1657
('with', 'the'): 1539
('of', 'a'): 1474


### Trigrams

In [9]:
trigrams = list(ngrams(clean_tokens, 3))
trigram_freq = Counter(trigrams)

print("Top 10 Trigrams:")
for triplet, freq in trigram_freq.most_common(10):
    print(f"{triplet}: {freq}")

Top 10 Trigrams:
('one', 'of', 'the'): 404
('the', 'united', 'states'): 336
('as', 'well', 'as'): 238
("''", 'he', 'said'): 222
('some', 'of', 'the'): 179
('out', 'of', 'the'): 174
('the', 'fact', 'that'): 167
('he', 'said', '``'): 155
('the', 'end', 'of'): 149
('part', 'of', 'the'): 144


## Next Word Prediction

### Input Sentence

In [10]:
partial_sentence = "The doctor said the patient might have to"
print("Input:", partial_sentence)

Input: The doctor said the patient might have to


### Statistical Language Model

In [11]:
import nltk
from nltk.corpus import brown
from nltk import FreqDist # NLTK specific frequency counting
from nltk import bigrams, trigrams, ngrams
from collections import defaultdict
import random

# Lowercase words from Brown corpus
tokens = [w.lower() for w in brown.words()]
clean_tokens = [w for w in tokens if w not in string.punctuation]
print("Done")

Done


In [12]:
# Build frequency distributions
bi_freq = FreqDist(bigrams(clean_tokens)) # a dictionary (bigram, count)
tri_freq = FreqDist(trigrams(clean_tokens)) # a dictionary (trigram, count)

print(f"Bigram frequency distribution sample: {bi_freq.most_common(5)}")
print(f"Trigram frequency distribution sample: {tri_freq.most_common(5)}")

# Get previous words of input
input_text = partial_sentence.lower().split()

last_input = tuple(input_text[-1:])
last_bigram = tuple(input_text[-2:])

print("=="*50)
print("Input: %s____"%partial_sentence)
print(f"Last input for Bigram: {last_input}")
print(f"Last bigram for Trigram: {last_bigram}")

Bigram frequency distribution sample: [(('of', 'the'), 9721), (('in', 'the'), 6041), (('to', 'the'), 3492), (('on', 'the'), 2477), (('and', 'the'), 2247)]
Trigram frequency distribution sample: [(('one', 'of', 'the'), 404), (('the', 'united', 'states'), 336), (('as', 'well', 'as'), 238), (("''", 'he', 'said'), 222), (('some', 'of', 'the'), 179)]
Input: The doctor said the patient might have to____
Last input for Bigram: ('to',)
Last bigram for Trigram: ('have', 'to')


In [13]:
# Get all trigrams that start with "as well"
item = [k for k,v in tri_freq.items() if k[:-1] == tuple(['as', 'well'])]
print(item)

[('as', 'well', 'as'), ('as', 'well', 'consequently'), ('as', 'well', 'to'), ('as', 'well', 'for'), ('as', 'well', 'meanwhile'), ('as', 'well', 'withdrawing'), ('as', 'well', 'politics-ridden'), ('as', 'well', 'stop'), ('as', 'well', 'it'), ('as', 'well', 'snow'), ('as', 'well', 'in'), ('as', 'well', 'was'), ('as', 'well', 'finally'), ('as', 'well', 'entries'), ('as', 'well', 'with'), ('as', 'well', 'under'), ('as', 'well', 'the'), ('as', 'well', 'informed'), ('as', 'well', '``'), ('as', 'well', "''"), ('as', 'well', 'but'), ('as', 'well', 'this'), ('as', 'well', 'many'), ('as', 'well', 'make'), ('as', 'well', 'along'), ('as', 'well', 'steinberg'), ('as', 'well', 'chance'), ('as', 'well', 'his'), ('as', 'well', 'pen'), ('as', 'well', 'similarly'), ('as', 'well', 'that'), ('as', 'well', 'thus'), ('as', 'well', 'also'), ('as', 'well', 'or'), ('as', 'well', 'having'), ('as', 'well', 'declines'), ('as', 'well', 'i'), ('as', 'well', 'seem'), ('as', 'well', 'far'), ('as', 'well', 'although')

In [14]:
# Get top 5 most likely next words that follow a given context
def get_most_likely_next_word(freq_dist, context):
    # the following line produces a dictionary of {next_word: count} for all matching n-grams
    candidates = {k[-1]: v for k, v in freq_dist.items() if k[:-1] == context}
    # Sort candidates by frequency (descending)
    sorted_candidates = sorted(candidates.items(), key=lambda x: -x[1])
    # Return top 5
    return sorted_candidates[:5]

bigram_predictions = get_most_likely_next_word(bi_freq, last_input)
print("Bigram prediction:")
print(bigram_predictions)

trigram_predictions = get_most_likely_next_word(tri_freq, last_bigram)
print("\nTrigram prediction:")
print(trigram_predictions)

Bigram prediction:
[('the', 3492), ('be', 1718), ('a', 660), ('do', 390), ('make', 354)]

Trigram prediction:
[('be', 57), ('do', 19), ('get', 9), ('take', 7), ('go', 7)]


In [15]:
print("Input Sentence Bigram Predictions:")
for word, count in bigram_predictions:
    sentence = f"{partial_sentence} {word}"
    print(f"{sentence}  [{count}]")

print("\nInput Sentence Trigram Predictions:")
for word, count in trigram_predictions:
    sentence = f"{partial_sentence} {word}"
    print(f"{sentence}  [{count}]")

Input Sentence Bigram Predictions:
The doctor said the patient might have to the  [3492]
The doctor said the patient might have to be  [1718]
The doctor said the patient might have to a  [660]
The doctor said the patient might have to do  [390]
The doctor said the patient might have to make  [354]

Input Sentence Trigram Predictions:
The doctor said the patient might have to be  [57]
The doctor said the patient might have to do  [19]
The doctor said the patient might have to get  [9]
The doctor said the patient might have to take  [7]
The doctor said the patient might have to go  [7]


## Transformer-based Language Model (Hugging Face GPT-2)
* GPT-2 is pretrained on massive web text (WebText corpus) in an unsupervised, causal language modeling fashion (i.e., predicting the next token given all previous ones).
* GPT-2 generates one token at a time, feeding each new token back into itself until it produces the **desired output length** or a **stopping condition** is met (like an end-of-sequence token).
* GPT-2 is **pretrained but not fine-tuned** for a specific task. This means it will continue generating words based on user's prompt.
* To control how many words/tokens it outputs we use **max_new_tokens**.
* For reproducibility we use **set_seed**.

In [17]:
# from transformers import pipeline, set_seed

# # Load text generation pipeline with GPT-2
# generator = pipeline("text-generation", model="gpt2")
# set_seed(42) # random seed for reproducibility

# prompt = "The doctor said the patient might have to" # → 8 tokens
# # generate a sequence of tokens (words, punctuation, etc.) up to 15 tokens total (including the prompt length)
# outputs = generator(prompt, max_length=15, num_return_sequences=3, max_new_tokens=1) # increase max_new_tokens

# print("\nGPT-2 Predictions:")
# for i, output in enumerate(outputs):
#     print(f"{i+1}: {output['generated_text']}")


### Final Notes:
* GPT-3 is not on Hugging Face since it is a proprietary model developed by OpenAI.
* How about Llama?
* Access to Llama models is restricted and **requires accepting the license on HF**.